In [0]:
# ============================================================================
# Pipeline Health Monitor — Salla E-Commerce Data Pipeline
# ============================================================================
# Runs post-pipeline to validate data quality, freshness, and row counts.
# Triggers email alerts (via job failure) on CRITICAL issues.
# ============================================================================

from datetime import datetime, timedelta
import json

# Pipeline IDs
PIPELINES = {
    "bronze": "cd8c7c88-a926-47e5-9580-f1934c44a037",
    "silver": "f5c1e5c9-8cc9-4c57-87fc-e0b40ba53803",
    "gold":   "62279bb1-c0a1-4670-ad61-0e07b1184d71",
}

# Expected minimum row counts (alert if below 50% of these)
EXPECTED_MINS = {
    # Bronze - PostgreSQL
    "bronze.bronze_customers": 9000,
    "bronze.bronze_products": 30,
    "bronze.bronze_stores": 40,
    "bronze.bronze_sales_orders": 1000000,
    "bronze.bronze_payment_transactions": 1000000,
    "bronze.bronze_shipping_details": 1000000,
    "bronze.bronze_inventory_movements": 400000,
    "bronze.bronze_product_reviews": 80000,
    # Bronze - Landing Zone
    "bronze.bronze_ad_spend": 30000,
    "bronze.bronze_competitor_pricing": 40000,
    "bronze.bronze_exchange_rates": 4000,
    "bronze.bronze_payment_settlements": 80000,
    "bronze.bronze_supplier_invoices": 200,
    "bronze.bronze_return_requests": 1000,
    "bronze.bronze_shipping_manifests": 150000,
    "bronze.bronze_clickstream": 1000000,
    "bronze.bronze_social_media": 100000,
    "bronze.bronze_push_notifications": 50000,
    "bronze.bronze_customer_segmentation": 9000,
    "bronze.bronze_demand_forecasts": 150000,
}

# Freshness SLA
FRESHNESS_WARNING_HOURS = 24
FRESHNESS_ALERT_HOURS = 48

# DQ violation threshold
DQ_ALERT_THRESHOLD_PCT = 5.0

# Store results for final summary
check_results = {"dq": [], "freshness": [], "row_counts": [], "filter_rates": []}

print(f"Pipeline Health Monitor initialized at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Freshness SLA: WARNING > {FRESHNESS_WARNING_HOURS}h, ALERT > {FRESHNESS_ALERT_HOURS}h")
print(f"DQ alert threshold: > {DQ_ALERT_THRESHOLD_PCT}% failure rate")

In [0]:
%sql
-- ============================================================================
-- DQ VIOLATIONS: Extract expectation pass/fail from pipeline event logs
-- ============================================================================

WITH bronze_events AS (
  SELECT
    timestamp,
    details
  FROM event_log('cd8c7c88-a926-47e5-9580-f1934c44a037')
  WHERE event_type = 'flow_progress'
    AND timestamp >= current_timestamp() - INTERVAL 24 HOURS
),
silver_events AS (
  SELECT
    timestamp,
    details
  FROM event_log('f5c1e5c9-8cc9-4c57-87fc-e0b40ba53803')
  WHERE event_type = 'flow_progress'
    AND timestamp >= current_timestamp() - INTERVAL 24 HOURS
),
all_events AS (
  SELECT 'bronze' AS layer, * FROM bronze_events
  UNION ALL
  SELECT 'silver' AS layer, * FROM silver_events
),
parsed AS (
  SELECT
    layer,
    timestamp,
    details:flow_progress.metrics.num_output_rows AS output_rows,
    explode(
      from_json(
        details:flow_progress:data_quality:expectations,
        'array<struct<name: string, dataset: string, passed_records: int, failed_records: int>>'
      )
    ) AS expectation
  FROM all_events
  WHERE details:flow_progress:data_quality:expectations IS NOT NULL
)
SELECT
  layer,
  expectation.dataset AS table_name,
  expectation.name AS expectation_name,
  SUM(expectation.passed_records) AS total_passed,
  SUM(expectation.failed_records) AS total_failed,
  ROUND(
    SUM(expectation.failed_records) * 100.0 /
    NULLIF(SUM(expectation.passed_records) + SUM(expectation.failed_records), 0),
    2
  ) AS failure_rate_pct,
  CASE
    WHEN SUM(expectation.failed_records) * 100.0 /
         NULLIF(SUM(expectation.passed_records) + SUM(expectation.failed_records), 0) > 5
    THEN '🔴 ALERT'
    WHEN SUM(expectation.failed_records) > 0
    THEN '🟡 WARNING'
    ELSE '🟢 PASS'
  END AS status
FROM parsed
GROUP BY layer, expectation.dataset, expectation.name
HAVING SUM(expectation.failed_records) > 0
ORDER BY failure_rate_pct DESC

In [0]:
%sql
-- ============================================================================
-- DATA FRESHNESS: Check last update timestamps for all landing zone tables
-- ============================================================================

WITH freshness AS (
  SELECT 'bronze_ad_spend' AS table_name, MAX(_bronze_timestamp) AS last_update FROM salla_databricks.bronze.bronze_ad_spend
  UNION ALL SELECT 'bronze_competitor_pricing', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_competitor_pricing
  UNION ALL SELECT 'bronze_exchange_rates', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_exchange_rates
  UNION ALL SELECT 'bronze_payment_settlements', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_payment_settlements
  UNION ALL SELECT 'bronze_supplier_invoices', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_supplier_invoices
  UNION ALL SELECT 'bronze_return_requests', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_return_requests
  UNION ALL SELECT 'bronze_shipping_manifests', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_shipping_manifests
  UNION ALL SELECT 'bronze_clickstream', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_clickstream
  UNION ALL SELECT 'bronze_social_media', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_social_media
  UNION ALL SELECT 'bronze_push_notifications', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_push_notifications
  UNION ALL SELECT 'bronze_customer_segmentation', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_customer_segmentation
  UNION ALL SELECT 'bronze_demand_forecasts', MAX(_bronze_timestamp) FROM salla_databricks.bronze.bronze_demand_forecasts
  UNION ALL SELECT 'silver_ad_spend', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_ad_spend
  UNION ALL SELECT 'silver_competitor_pricing', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_competitor_pricing
  UNION ALL SELECT 'silver_exchange_rates', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_exchange_rates
  UNION ALL SELECT 'silver_payment_settlements', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_payment_settlements
  UNION ALL SELECT 'silver_supplier_invoices', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_supplier_invoices
  UNION ALL SELECT 'silver_return_requests', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_return_requests
  UNION ALL SELECT 'silver_shipping_manifests', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_shipping_manifests
  UNION ALL SELECT 'silver_clickstream', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_clickstream
  UNION ALL SELECT 'silver_social_media', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_social_media
  UNION ALL SELECT 'silver_push_notifications', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_push_notifications
  UNION ALL SELECT 'silver_customer_segmentation', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_customer_segmentation
  UNION ALL SELECT 'silver_demand_forecasts', MAX(_silver_timestamp) FROM salla_databricks.silver.silver_demand_forecasts
)
SELECT
  table_name,
  last_update,
  ROUND((unix_timestamp(current_timestamp()) - unix_timestamp(last_update)) / 3600, 1) AS hours_since_update,
  CASE
    WHEN last_update IS NULL THEN '🔴 ALERT - No data'
    WHEN (unix_timestamp(current_timestamp()) - unix_timestamp(last_update)) / 3600 > 48 THEN '🔴 ALERT - Stale >48h'
    WHEN (unix_timestamp(current_timestamp()) - unix_timestamp(last_update)) / 3600 > 24 THEN '🟡 WARNING - Stale >24h'
    ELSE '🟢 FRESH'
  END AS status
FROM freshness
ORDER BY hours_since_update DESC

In [0]:
%sql
-- ============================================================================
-- ROW COUNTS: Validate all 48 tables have expected minimum rows
-- ============================================================================

WITH counts AS (
  -- Bronze PostgreSQL (8)
  SELECT 'bronze' AS layer, 'bronze_customers' AS table_name, COUNT(*) AS row_count, 9000 AS expected_min FROM salla_databricks.bronze.bronze_customers
  UNION ALL SELECT 'bronze', 'bronze_products', COUNT(*), 30 FROM salla_databricks.bronze.bronze_products
  UNION ALL SELECT 'bronze', 'bronze_stores', COUNT(*), 40 FROM salla_databricks.bronze.bronze_stores
  UNION ALL SELECT 'bronze', 'bronze_sales_orders', COUNT(*), 1000000 FROM salla_databricks.bronze.bronze_sales_orders
  UNION ALL SELECT 'bronze', 'bronze_payment_transactions', COUNT(*), 1000000 FROM salla_databricks.bronze.bronze_payment_transactions
  UNION ALL SELECT 'bronze', 'bronze_shipping_details', COUNT(*), 1000000 FROM salla_databricks.bronze.bronze_shipping_details
  UNION ALL SELECT 'bronze', 'bronze_inventory_movements', COUNT(*), 400000 FROM salla_databricks.bronze.bronze_inventory_movements
  UNION ALL SELECT 'bronze', 'bronze_product_reviews', COUNT(*), 80000 FROM salla_databricks.bronze.bronze_product_reviews
  -- Bronze Landing Zone (12)
  UNION ALL SELECT 'bronze', 'bronze_ad_spend', COUNT(*), 30000 FROM salla_databricks.bronze.bronze_ad_spend
  UNION ALL SELECT 'bronze', 'bronze_competitor_pricing', COUNT(*), 40000 FROM salla_databricks.bronze.bronze_competitor_pricing
  UNION ALL SELECT 'bronze', 'bronze_exchange_rates', COUNT(*), 4000 FROM salla_databricks.bronze.bronze_exchange_rates
  UNION ALL SELECT 'bronze', 'bronze_payment_settlements', COUNT(*), 80000 FROM salla_databricks.bronze.bronze_payment_settlements
  UNION ALL SELECT 'bronze', 'bronze_supplier_invoices', COUNT(*), 200 FROM salla_databricks.bronze.bronze_supplier_invoices
  UNION ALL SELECT 'bronze', 'bronze_return_requests', COUNT(*), 1000 FROM salla_databricks.bronze.bronze_return_requests
  UNION ALL SELECT 'bronze', 'bronze_shipping_manifests', COUNT(*), 150000 FROM salla_databricks.bronze.bronze_shipping_manifests
  UNION ALL SELECT 'bronze', 'bronze_clickstream', COUNT(*), 1000000 FROM salla_databricks.bronze.bronze_clickstream
  UNION ALL SELECT 'bronze', 'bronze_social_media', COUNT(*), 100000 FROM salla_databricks.bronze.bronze_social_media
  UNION ALL SELECT 'bronze', 'bronze_push_notifications', COUNT(*), 50000 FROM salla_databricks.bronze.bronze_push_notifications
  UNION ALL SELECT 'bronze', 'bronze_customer_segmentation', COUNT(*), 9000 FROM salla_databricks.bronze.bronze_customer_segmentation
  UNION ALL SELECT 'bronze', 'bronze_demand_forecasts', COUNT(*), 150000 FROM salla_databricks.bronze.bronze_demand_forecasts
  -- Silver (20)
  UNION ALL SELECT 'silver', 'silver_customers', COUNT(*), 9000 FROM salla_databricks.silver.silver_customers
  UNION ALL SELECT 'silver', 'silver_products', COUNT(*), 30 FROM salla_databricks.silver.silver_products
  UNION ALL SELECT 'silver', 'silver_stores', COUNT(*), 40 FROM salla_databricks.silver.silver_stores
  UNION ALL SELECT 'silver', 'silver_sales_orders', COUNT(*), 1000000 FROM salla_databricks.silver.silver_sales_orders
  UNION ALL SELECT 'silver', 'silver_payment_transactions', COUNT(*), 1000000 FROM salla_databricks.silver.silver_payment_transactions
  UNION ALL SELECT 'silver', 'silver_shipping_details', COUNT(*), 1000000 FROM salla_databricks.silver.silver_shipping_details
  UNION ALL SELECT 'silver', 'silver_inventory_movements', COUNT(*), 400000 FROM salla_databricks.silver.silver_inventory_movements
  UNION ALL SELECT 'silver', 'silver_product_reviews', COUNT(*), 80000 FROM salla_databricks.silver.silver_product_reviews
  UNION ALL SELECT 'silver', 'silver_ad_spend', COUNT(*), 25000 FROM salla_databricks.silver.silver_ad_spend
  UNION ALL SELECT 'silver', 'silver_competitor_pricing', COUNT(*), 4000 FROM salla_databricks.silver.silver_competitor_pricing
  UNION ALL SELECT 'silver', 'silver_exchange_rates', COUNT(*), 3500 FROM salla_databricks.silver.silver_exchange_rates
  UNION ALL SELECT 'silver', 'silver_payment_settlements', COUNT(*), 75000 FROM salla_databricks.silver.silver_payment_settlements
  UNION ALL SELECT 'silver', 'silver_supplier_invoices', COUNT(*), 180 FROM salla_databricks.silver.silver_supplier_invoices
  UNION ALL SELECT 'silver', 'silver_return_requests', COUNT(*), 900 FROM salla_databricks.silver.silver_return_requests
  UNION ALL SELECT 'silver', 'silver_shipping_manifests', COUNT(*), 140000 FROM salla_databricks.silver.silver_shipping_manifests
  UNION ALL SELECT 'silver', 'silver_clickstream', COUNT(*), 900000 FROM salla_databricks.silver.silver_clickstream
  UNION ALL SELECT 'silver', 'silver_social_media', COUNT(*), 90000 FROM salla_databricks.silver.silver_social_media
  UNION ALL SELECT 'silver', 'silver_push_notifications', COUNT(*), 45000 FROM salla_databricks.silver.silver_push_notifications
  UNION ALL SELECT 'silver', 'silver_customer_segmentation', COUNT(*), 9000 FROM salla_databricks.silver.silver_customer_segmentation
  UNION ALL SELECT 'silver', 'silver_demand_forecasts', COUNT(*), 140000 FROM salla_databricks.silver.silver_demand_forecasts
  -- Gold (8)
  UNION ALL SELECT 'gold', 'dim_customers', COUNT(*), 9000 FROM salla_databricks.gold.dim_customers
  UNION ALL SELECT 'gold', 'dim_products', COUNT(*), 30 FROM salla_databricks.gold.dim_products
  UNION ALL SELECT 'gold', 'dim_stores', COUNT(*), 40 FROM salla_databricks.gold.dim_stores
  UNION ALL SELECT 'gold', 'dim_date', COUNT(*), 2000 FROM salla_databricks.gold.dim_date
  UNION ALL SELECT 'gold', 'dim_payment_method', COUNT(*), 5 FROM salla_databricks.gold.dim_payment_method
  UNION ALL SELECT 'gold', 'fact_sales', COUNT(*), 1000000 FROM salla_databricks.gold.fact_sales
  UNION ALL SELECT 'gold', 'fact_ad_spend', COUNT(*), 25000 FROM salla_databricks.gold.fact_ad_spend
  UNION ALL SELECT 'gold', 'fact_competitor_pricing', COUNT(*), 4000 FROM salla_databricks.gold.fact_competitor_pricing
)
SELECT
  layer,
  table_name,
  row_count,
  expected_min,
  ROUND(row_count * 100.0 / NULLIF(expected_min, 0), 1) AS pct_of_expected,
  CASE
    WHEN row_count = 0 THEN '🔴 ALERT - Empty table'
    WHEN row_count < expected_min * 0.5 THEN '🔴 ALERT - Below 50%'
    WHEN row_count < expected_min THEN '🟡 WARNING - Below expected'
    ELSE '🟢 PASS'
  END AS status
FROM counts
ORDER BY
  CASE WHEN row_count = 0 THEN 0 WHEN row_count < expected_min * 0.5 THEN 1 ELSE 2 END,
  pct_of_expected ASC

In [0]:
%sql
-- ============================================================================
-- DQ FILTER RATE: Compare Bronze vs Silver to measure DQ effectiveness
-- ============================================================================

WITH bronze_counts AS (
  SELECT 'ad_spend' AS source, COUNT(*) AS bronze_rows FROM salla_databricks.bronze.bronze_ad_spend
  UNION ALL SELECT 'competitor_pricing', COUNT(*) FROM salla_databricks.bronze.bronze_competitor_pricing
  UNION ALL SELECT 'exchange_rates', COUNT(*) FROM salla_databricks.bronze.bronze_exchange_rates
  UNION ALL SELECT 'payment_settlements', COUNT(*) FROM salla_databricks.bronze.bronze_payment_settlements
  UNION ALL SELECT 'supplier_invoices', COUNT(*) FROM salla_databricks.bronze.bronze_supplier_invoices
  UNION ALL SELECT 'return_requests', COUNT(*) FROM salla_databricks.bronze.bronze_return_requests
  UNION ALL SELECT 'shipping_manifests', COUNT(*) FROM salla_databricks.bronze.bronze_shipping_manifests
  UNION ALL SELECT 'clickstream', COUNT(*) FROM salla_databricks.bronze.bronze_clickstream
  UNION ALL SELECT 'social_media', COUNT(*) FROM salla_databricks.bronze.bronze_social_media
  UNION ALL SELECT 'push_notifications', COUNT(*) FROM salla_databricks.bronze.bronze_push_notifications
  UNION ALL SELECT 'customer_segmentation', COUNT(*) FROM salla_databricks.bronze.bronze_customer_segmentation
  UNION ALL SELECT 'demand_forecasts', COUNT(*) FROM salla_databricks.bronze.bronze_demand_forecasts
),
silver_counts AS (
  SELECT 'ad_spend' AS source, COUNT(*) AS silver_rows FROM salla_databricks.silver.silver_ad_spend
  UNION ALL SELECT 'competitor_pricing', COUNT(*) FROM salla_databricks.silver.silver_competitor_pricing
  UNION ALL SELECT 'exchange_rates', COUNT(*) FROM salla_databricks.silver.silver_exchange_rates
  UNION ALL SELECT 'payment_settlements', COUNT(*) FROM salla_databricks.silver.silver_payment_settlements
  UNION ALL SELECT 'supplier_invoices', COUNT(*) FROM salla_databricks.silver.silver_supplier_invoices
  UNION ALL SELECT 'return_requests', COUNT(*) FROM salla_databricks.silver.silver_return_requests
  UNION ALL SELECT 'shipping_manifests', COUNT(*) FROM salla_databricks.silver.silver_shipping_manifests
  UNION ALL SELECT 'clickstream', COUNT(*) FROM salla_databricks.silver.silver_clickstream
  UNION ALL SELECT 'social_media', COUNT(*) FROM salla_databricks.silver.silver_social_media
  UNION ALL SELECT 'push_notifications', COUNT(*) FROM salla_databricks.silver.silver_push_notifications
  UNION ALL SELECT 'customer_segmentation', COUNT(*) FROM salla_databricks.silver.silver_customer_segmentation
  UNION ALL SELECT 'demand_forecasts', COUNT(*) FROM salla_databricks.silver.silver_demand_forecasts
)
SELECT
  b.source,
  b.bronze_rows,
  s.silver_rows,
  b.bronze_rows - s.silver_rows AS rows_filtered,
  ROUND((1 - s.silver_rows * 1.0 / NULLIF(b.bronze_rows, 0)) * 100, 1) AS filter_pct,
  CASE
    WHEN b.bronze_rows = 0 THEN '🔴 ALERT - No data'
    WHEN b.source = 'competitor_pricing' AND (1 - s.silver_rows * 1.0 / b.bronze_rows) * 100 > 80
      THEN '🟢 EXPECTED - Strict price validation'
    WHEN (1 - s.silver_rows * 1.0 / NULLIF(b.bronze_rows, 0)) * 100 > 50
      THEN '🔴 ALERT - Excessive filtering'
    WHEN (1 - s.silver_rows * 1.0 / NULLIF(b.bronze_rows, 0)) * 100 > 20
      THEN '🟡 WARNING - High filter rate'
    ELSE '🟢 NORMAL'
  END AS status
FROM bronze_counts b
JOIN silver_counts s ON b.source = s.source
ORDER BY filter_pct DESC

In [0]:
# ============================================================================
# SUMMARY & ALERTING: Aggregate all checks, fail job on CRITICAL issues
# ============================================================================

from datetime import datetime
import json

# Run summary queries to collect results
print("="*70)
print("PIPELINE HEALTH CHECK SUMMARY")
print(f"Run time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

critical_issues = []
warnings = []

# 1. Check for empty tables (CRITICAL)
empty_tables = spark.sql("""
  SELECT table_name, row_count FROM (
    SELECT 'bronze_sales_orders' AS table_name, COUNT(*) AS row_count FROM salla_databricks.bronze.bronze_sales_orders
    UNION ALL SELECT 'silver_sales_orders', COUNT(*) FROM salla_databricks.silver.silver_sales_orders
    UNION ALL SELECT 'gold.fact_sales', COUNT(*) FROM salla_databricks.gold.fact_sales
    UNION ALL SELECT 'bronze_clickstream', COUNT(*) FROM salla_databricks.bronze.bronze_clickstream
  ) WHERE row_count = 0
""").collect()

for row in empty_tables:
    critical_issues.append(f"EMPTY TABLE: {row.table_name}")

# 2. Check data freshness (WARNING if >24h, CRITICAL if >48h)
freshness_check = spark.sql("""
  SELECT table_name, hours_since FROM (
    SELECT 'bronze_clickstream' AS table_name,
           ROUND((unix_timestamp(current_timestamp()) - unix_timestamp(MAX(_bronze_timestamp))) / 3600, 1) AS hours_since
    FROM salla_databricks.bronze.bronze_clickstream
    UNION ALL
    SELECT 'bronze_sales_orders',
           ROUND((unix_timestamp(current_timestamp()) - unix_timestamp(MAX(_bronze_timestamp))) / 3600, 1)
    FROM salla_databricks.bronze.bronze_sales_orders
  ) WHERE hours_since > 24
""").collect()

for row in freshness_check:
    if row.hours_since and row.hours_since > 48:
        critical_issues.append(f"STALE DATA (>48h): {row.table_name} - {row.hours_since}h")
    elif row.hours_since and row.hours_since > 24:
        warnings.append(f"Stale data (>24h): {row.table_name} - {row.hours_since}h")

# 3. Check row counts below 50% threshold
low_count_tables = spark.sql("""
  SELECT * FROM (
    SELECT 'fact_sales' AS table_name, COUNT(*) AS cnt, 1000000 AS expected FROM salla_databricks.gold.fact_sales
    UNION ALL SELECT 'bronze_clickstream', COUNT(*), 1000000 FROM salla_databricks.bronze.bronze_clickstream
    UNION ALL SELECT 'silver_clickstream', COUNT(*), 900000 FROM salla_databricks.silver.silver_clickstream
  ) WHERE cnt < expected * 0.5
""").collect()

for row in low_count_tables:
    critical_issues.append(f"LOW ROW COUNT: {row.table_name} has {row.cnt:,} rows (expected {row.expected:,})")

# 4. Check for abnormal DQ filter rates (excluding competitor_pricing which is expected high)
filter_check = spark.sql("""
  WITH b AS (SELECT 'social_media' AS src, COUNT(*) AS cnt FROM salla_databricks.bronze.bronze_social_media
             UNION ALL SELECT 'clickstream', COUNT(*) FROM salla_databricks.bronze.bronze_clickstream),
       s AS (SELECT 'social_media' AS src, COUNT(*) AS cnt FROM salla_databricks.silver.silver_social_media
             UNION ALL SELECT 'clickstream', COUNT(*) FROM salla_databricks.silver.silver_clickstream)
  SELECT b.src, ROUND((1 - s.cnt * 1.0 / b.cnt) * 100, 1) AS filter_pct
  FROM b JOIN s ON b.src = s.src
  WHERE (1 - s.cnt * 1.0 / b.cnt) * 100 > 50
""").collect()

for row in filter_check:
    critical_issues.append(f"EXCESSIVE DQ FILTERING: {row.src} - {row.filter_pct}% filtered")

# Print summary
print("\n" + "-"*70)
print("CRITICAL ISSUES:")
if critical_issues:
    for issue in critical_issues:
        print(f"  🔴 {issue}")
else:
    print("  🟢 None")

print("\nWARNINGS:")
if warnings:
    for warn in warnings:
        print(f"  🟡 {warn}")
else:
    print("  🟢 None")

print("-"*70)

# Build result JSON
result = {
    "status": "FAIL" if critical_issues else "PASS",
    "timestamp": datetime.now().isoformat(),
    "critical_count": len(critical_issues),
    "warning_count": len(warnings),
    "critical_issues": critical_issues,
    "warnings": warnings,
}

print(f"\nOVERALL STATUS: {'\u274c FAIL' if critical_issues else '\u2705 PASS'}")
print("="*70)

# Exit with result JSON (for downstream consumption)
if critical_issues:
    # Raise exception to fail the job task, triggering email notification
    dbutils.notebook.exit(json.dumps(result))
    raise Exception(f"Pipeline health check FAILED with {len(critical_issues)} critical issue(s): {'; '.join(critical_issues)}")
else:
    dbutils.notebook.exit(json.dumps(result))